# Gemma 3 12B — Merge LoRA + Export for TRT-LLM INT4

**Purpose**: Merge your trained LoRA adapter with the base model, then export for TensorRT-LLM INT4 engine build.

**What you need**: Your LoRA adapter directory (already on Colab, Google Drive, or upload zip)

**What you get**: Merged 16-bit model (~24GB) ready for INT4 TRT engine conversion

**Hardware**: A100 (40GB) recommended. T4 (15GB) works but slower.

**Time**: ~30 minutes total

---

## Your LoRA Adapter Details
- **Base model**: `unsloth/gemma-3-12b-it-unsloth-bnb-4bit`
- **Architecture**: `Gemma3ForConditionalGeneration` (text + vision)
- **LoRA**: r=16, alpha=32, rslora=True, 7 target modules
- **Training**: 3750 steps, 2 epochs, checkpoint from March 8 2026
- **Adapter size**: ~262MB (adapter_model.safetensors)

## 1. Install Dependencies

In [ ]:
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft transformers huggingface_hub

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

## 2. Find or Upload LoRA Adapter

This cell checks three places in order:
1. Already on this Colab instance (from a previous training run)
2. On Google Drive
3. Upload zip from local machine

In [ ]:
import os
import shutil
from pathlib import Path

# Known adapter directory names (from training notebooks)
ADAPTER_CANDIDATES = [
    'gemma3-ckpt3750vlmtrained',
    'gemma3-12b-legal-multimodal-lora',
    'gemma3-12b-legal-ace-synthesis-lora',
    'gemma3-legal-lora',
    'gemma3n-legal-lora',
    # Training output dirs (checkpoints)
    'gemma3-12b-legal-multimodal/checkpoint-3750',
    'gemma3-12b-ace-synthesis/checkpoint-3750',
    'gemma3n-legal-outputs/checkpoint-3750',
]

ADAPTER_DIR = None

# --- Step 1: Check current Colab filesystem ---
print("Step 1: Checking Colab filesystem...")
for candidate in ADAPTER_CANDIDATES:
    full_path = Path(candidate)
    # Also check /content/ prefix
    for prefix in ['', '/content/']:
        p = Path(prefix + candidate)
        if p.exists() and (p / 'adapter_config.json').exists():
            ADAPTER_DIR = str(p)
            print(f"  FOUND: {ADAPTER_DIR}")
            break
    if ADAPTER_DIR:
        break

# --- Step 2: Check Google Drive ---
if not ADAPTER_DIR:
    print("  Not found locally.")
    print("\nStep 2: Checking Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        
        drive_candidates = [
            '/content/drive/MyDrive/' + c for c in ADAPTER_CANDIDATES
        ] + [
            '/content/drive/MyDrive/COLAB_PACKAGE/' + c for c in ADAPTER_CANDIDATES
        ]
        
        for candidate in drive_candidates:
            p = Path(candidate)
            if p.exists() and (p / 'adapter_config.json').exists():
                # Copy to local for faster access
                local_name = p.name
                print(f"  Found on Drive: {candidate}")
                print(f"  Copying to local...")
                shutil.copytree(str(p), local_name, dirs_exist_ok=True)
                ADAPTER_DIR = local_name
                break
        
        # Also check for zip files on Drive
        if not ADAPTER_DIR:
            for zip_name in ['gemma3-ckpt3750vlmtrained.zip', 'gemma3-legal-lora-backup.zip',
                             'gemma3-12b-legal-multimodal-lora.zip']:
                zip_path = Path(f'/content/drive/MyDrive/{zip_name}')
                if zip_path.exists():
                    print(f"  Found zip on Drive: {zip_path}")
                    !unzip -o "{zip_path}" -d /content/
                    # Re-scan for adapter
                    for candidate in ADAPTER_CANDIDATES:
                        p = Path(candidate)
                        if p.exists() and (p / 'adapter_config.json').exists():
                            ADAPTER_DIR = str(p)
                            break
                    if ADAPTER_DIR:
                        break
    except Exception as e:
        print(f"  Drive not available: {e}")

# --- Step 3: Manual upload ---
if not ADAPTER_DIR:
    print("  Not found on Drive.")
    print("\nStep 3: Upload adapter zip manually...")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            !unzip -o "{name}"
            print(f"  Extracted: {name}")
    
    for candidate in ADAPTER_CANDIDATES:
        p = Path(candidate)
        if p.exists() and (p / 'adapter_config.json').exists():
            ADAPTER_DIR = str(p)
            break

# --- Result ---
print(f"\n{'='*60}")
if ADAPTER_DIR:
    print(f"Adapter: {ADAPTER_DIR}/")
    for f in sorted(Path(ADAPTER_DIR).iterdir()):
        if not f.name.startswith('.'):
            size = f.stat().st_size / (1024**2)
            print(f"  {f.name:40s} ({size:>6.1f} MB)")
else:
    raise FileNotFoundError(
        "No adapter found! Expected a directory containing:\n"
        "  - adapter_config.json\n"
        "  - adapter_model.safetensors\n"
        "Upload the zip or copy to Google Drive first."
    )

## 3. Load Base Model + Merge LoRA

Unsloth's `FastVisionModel.from_pretrained()` auto-detects adapter directories
and loads base model + LoRA together. Then `save_pretrained_merged()` bakes
the LoRA weights into the base model.

In [ ]:
import json
from unsloth import FastVisionModel, is_bfloat16_supported

# Read adapter config for info
with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    adapter_config = json.load(f)

base_model_name = adapter_config.get('base_model_name_or_path', 'unknown')
auto_mapping = adapter_config.get('auto_mapping', {})
arch_class = auto_mapping.get('base_model_class', 'unknown')

print(f"Base model: {base_model_name}")
print(f"Architecture: {arch_class}")
print(f"LoRA rank: {adapter_config.get('r', '?')}")
print(f"LoRA alpha: {adapter_config.get('lora_alpha', '?')}")
print(f"RSLoRA: {adapter_config.get('use_rslora', False)}")
print(f"Target modules: {adapter_config.get('target_modules', [])}")
print(f"\nLoading model + adapter...\n")

# Load directly from adapter dir — Unsloth auto-loads base + adapter
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

print(f"\nModel loaded successfully from {ADAPTER_DIR}")

## 4. Quick Inference Test (Optional)

Verify the adapter works before merging. Skip this cell if you want to go straight to merge.

In [ ]:
from unsloth.chat_templates import get_chat_template

FastVisionModel.for_inference(model)

# Ensure Gemma 3 chat template is applied
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

test_prompts = [
    "What is the legal standard for admissibility of evidence under the Federal Rules of Evidence?",
    "Explain how Svelte 5 runes ($state, $derived, $effect) replace Svelte 4 stores.",
]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"Q: {prompt}")
    print(f"{'='*70}")
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    model.generate(
        input_ids=inputs, streamer=streamer,
        max_new_tokens=200, temperature=0.7, top_p=0.9, use_cache=True
    )
    print()

## 5. Merge LoRA into Base Model (16-bit)

This creates a standalone model with LoRA weights baked in.
Output: ~24GB for Gemma 3 12B in 16-bit.

**Important**: `save_pretrained_merged` is an Unsloth method.
It only works when the model was loaded via `FastVisionModel.from_pretrained()`.

In [ ]:
from pathlib import Path

MERGED_DIR = "gemma3-12b-legal-merged-16bit"

print(f"Merging LoRA weights into base model...")
print(f"Output: {MERGED_DIR}/ (~24 GB)")
print(f"This takes 10-15 minutes on A100.\n")

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit"
)

# Verify output
merged_path = Path(MERGED_DIR)
total_gb = sum(f.stat().st_size for f in merged_path.rglob('*') if f.is_file()) / (1024**3)
safetensor_files = sorted(merged_path.glob('*.safetensors'))

print(f"\nMerged model saved: {MERGED_DIR}/")
print(f"Total size: {total_gb:.1f} GB")
print(f"Safetensor shards: {len(safetensor_files)}")
for f in safetensor_files:
    print(f"  {f.name}: {f.stat().st_size / (1024**3):.1f} GB")

## 6. Verify Merged Model Config

Check that the merged model has the right architecture for TRT-LLM conversion.

In [ ]:
import json
from pathlib import Path

config_path = Path(MERGED_DIR) / 'config.json'
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    
    print("Merged Model Config:")
    print(f"  Architecture: {config.get('architectures', ['?'])[0]}")
    print(f"  Hidden size: {config.get('hidden_size', '?')}")
    print(f"  Num layers: {config.get('num_hidden_layers', '?')}")
    print(f"  Num heads: {config.get('num_attention_heads', '?')}")
    print(f"  Vocab size: {config.get('vocab_size', '?')}")
    print(f"  Max position: {config.get('max_position_embeddings', '?')}")
    
    # Check for VLM components
    arch = config.get('architectures', [''])[0]
    if 'Conditional' in arch:
        print(f"\n  VLM: YES ({arch})")
        text_config = config.get('text_config', {})
        print(f"  Text decoder: hidden_size={text_config.get('hidden_size', config.get('hidden_size', '?'))}")
        print(f"  Text layers: {text_config.get('num_hidden_layers', config.get('num_hidden_layers', '?'))}")
        if 'vision_config' in config:
            vc = config['vision_config']
            print(f"  Vision encoder: {vc.get('model_type', '?')} ({vc.get('hidden_size', '?')}-dim)")
            print(f"  Vision image size: {vc.get('image_size', '?')}")
        print(f"\n  For TRT-LLM: extract text decoder only (vision stays PyTorch)")
    else:
        print(f"\n  VLM: NO (text-only: {arch})")
        print(f"\n  For TRT-LLM: convert entire model")
    
    print(f"\n  TRT-LLM conversion command:")
    print(f"    python convert_checkpoint.py \\")
    print(f"      --model_dir {MERGED_DIR} \\")
    print(f"      --output_dir trt_checkpoint \\")
    print(f"      --dtype float16 \\")
    print(f"      --tp_size 1")
else:
    print("WARNING: config.json not found in merged model!")

## 7. Save to Google Drive

The merged model is ~24GB. Google Drive is the most reliable way to get it off Colab.

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

drive_dest = '/content/drive/MyDrive/gemma3-12b-legal-merged-16bit'

print(f"Copying to Google Drive: {drive_dest}")
print(f"This takes 10-20 minutes for ~24GB...\n")

if os.path.exists(drive_dest):
    print(f"Removing existing: {drive_dest}")
    shutil.rmtree(drive_dest)

shutil.copytree(MERGED_DIR, drive_dest)

# Verify
drive_size = sum(f.stat().st_size for f in Path(drive_dest).rglob('*') if f.is_file()) / (1024**3)
print(f"\nSaved to Google Drive: {drive_size:.1f} GB")
print(f"Download from: https://drive.google.com/")
print(f"Path: My Drive / gemma3-12b-legal-merged-16bit/")
print(f"\nTip: Download as zip from Drive for faster transfer")

## 8. (Optional) Push to HuggingFace Hub

Alternative to Google Drive — push to a private HF repo for easy `git clone` later.

In [ ]:
# Uncomment all lines below to use:

# from huggingface_hub import notebook_login
# notebook_login()
#
# HF_REPO = "YOUR_USERNAME/gemma3-12b-legal-merged"  # Change this!
#
# model.push_to_hub_merged(
#     HF_REPO,
#     tokenizer,
#     save_method="merged_16bit",
#     private=True,
# )
#
# print(f"Pushed to: https://huggingface.co/{HF_REPO}")
# print(f"Clone locally: git clone https://huggingface.co/{HF_REPO}")

## 9. Next Steps (On Your Local Machine)

After downloading the merged model (~24GB):

### Text Inference: TRT-LLM INT4 Engine
```bash
# 1. Place merged model in project
mkdir -p ~/gemma3-12b-legal-merged-16bit
# Copy/extract downloaded files here

# 2. Build INT4 engine inside Docker container
cd ~/Videos/deeds-web-app
bash scripts/build-trt-engine-in-container.sh

# 3. Start Triton
docker compose -f docker-compose.triton.yml up -d
```

### Vision Inference: PyTorch in TRT-LLM Container
The merged model is `Gemma3ForConditionalGeneration` (VLM).
The TRT-LLM v0.21.0 container has PyTorch + transformers built-in.
For image queries, load the full VLM via PyTorch on-demand:

```python
# Inside TRT-LLM container:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

model = Gemma3ForConditionalGeneration.from_pretrained(
    '/models/gemma3-12b-legal-merged-16bit',
    torch_dtype=torch.float16,
    device_map='auto'
)
processor = AutoProcessor.from_pretrained('/models/gemma3-12b-legal-merged-16bit')

# Process image + text
inputs = processor(images=image, text=prompt, return_tensors='pt').to('cuda')
output = model.generate(**inputs, max_new_tokens=256)

# Unload to free VRAM for TRT engine
del model, processor
torch.cuda.empty_cache()
```

### Architecture: Text/Vision Switcher
```
Request -> /api/ai/tensorrt/
  |-- text-only -> TRT INT4 Engine (fast, persistent, ~6.5GB)
  `-- image+text -> PyTorch VLM (on-demand, load/unload)
```

Your SvelteKit API route at `src/routes/api/ai/tensorrt/vlm/+server.ts`
already implements this pattern with GPU lease acquisition and model load/unload.